In [1]:
# Importações e setup

#from selenium import webdriver
from seleniumwire import webdriver
from selenium.webdriver.support.select import Select
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.common.by import By
from selenium.webdriver.common.action_chains import ActionChains
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.common.alert import Alert
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.chrome.options import Options
from selenium.common.exceptions import NoSuchElementException
from selenium.common.exceptions import TimeoutException, WebDriverException
from webdriver_manager.chrome import ChromeDriverManager
from selenium.common.exceptions import ElementClickInterceptedException

#Bibliotecas de Sistema
from datetime import datetime
from datetime import timedelta
from datetime import date
from datetime import timezone
import time
import re
import csv
import os
import requests
import psutil
from bs4 import BeautifulSoup
from eproc_driver import eproc as eproc
import sqlite3
from pathlib import Path
import io
import pandas as pd
from contextlib import closing
from sympy import false
from pydoc import text
from sympy import true
import PyPDF2
import glob

#bibliotecas de configuração
import pyotp
import configparser
import keyring

#bibliotecas de automação
import pyperclip
import pyautogui

#Bibliotecas de IA
from gemini import gemini as gemini
import ollama

#pasta_downloads = r"C:\Users\dodonin\Downloads"
pasta_downloads = r"D:\Douglas\Reps\UNICA\PDFs"

# DEBUG, sim ou não
debug = False

In [2]:
# Definição de perfil e tipo de processo

#perfil = "SRD1CIV"
#perfil = "CAN1CIV"
#perfil = "GDO1CIV"
perfil = "STM1CIV"
#perfil = "SRO2CR"
#perfil = "POA08FZFC"

tipo_processo = "EXECUÇÃO FISCAL"

print("Perfil escolhido:", perfil)
print("Tipo de Processo:", tipo_processo)

Perfil escolhido: STM1CIV
Tipo de Processo: EXECUÇÃO FISCAL


In [5]:
# Define funções

def pega_tabela_pagina(dados_tabela):
    tabela = navegador.find_element(By.ID, "tabelaLocalizadores")
    linhas = tabela.find_elements(By.TAG_NAME, "tr")[1:]  # Ignora o cabeçalho
    lastpage = false
    while lastpage == false:
        for linha in linhas:
            # Aguarda o carregamento do tbody da tabela antes de processar as linhas
            WebDriverWait(navegador, 10).until(
                EC.presence_of_element_located((By.XPATH, "//table[@id='tabelaLocalizadores']/tbody"))
            )
            colunas = linha.find_elements(By.TAG_NAME, "td")
            if len(colunas) >= 4:
                # Pega as três primeiras colunas e também a última
                dados_tabela.append([
                    colunas[1].text.split('\n')[0],
                    colunas[2].text.split('\n')[0],
                    colunas[3].text.split('\n')[0],
                    colunas[-1].text.split('\n')[0]
                ])
            
            lastpage = true

def pega_ultima_peticao(navegador, num_processo):
    """
    Localiza a última petição ("PET") disponível no processo informado,
    captura a requisição real feita pelo navegador (Selenium Wire) para baixar o PDF
    e salva o arquivo localmente.

    Parâmetros:
        navegador     -> instância do Selenium Wire WebDriver já autenticada no eproc.
        num_processo  -> número do processo (será usado no nome do PDF).

    Retorna:
        (evento_id, caminho_pdf) se o download foi bem-sucedido.
        (None, None) caso não haja petição ou falha no download.
    """
    import os
    import requests
    from selenium.webdriver.common.by import By
    import glob

    # === 1. Coleta todos os eventos do processo ===
    documentos_eventos = []
    eventos = navegador.find_elements(By.CLASS_NAME, "td-evento")

    for evento in eventos:
        # ID do elemento que identifica o documento
        doc_id = evento.get_dom_attribute("id")

        # Sobe até a linha da tabela e pega o número do evento
        tr_element = evento.find_element(By.XPATH, "./ancestor::tr")
        evento_id = tr_element.find_element(By.XPATH, './td[2]').text
        evento_id = ''.join(filter(str.isdigit, evento_id))  # mantém só números

        # Verifica se é documento tipo PET (Petição)
        try:
            infra_link = evento.find_element(By.XPATH, ".//*[contains(@class, 'infraLinkDoc')]")
            data_nome = infra_link.get_attribute("data-nome")
        except Exception:
            data_nome = None

        documentos_eventos.append((doc_id, data_nome, evento_id))

    # === 2. Filtra apenas documentos do tipo PET ===
    documentos_eventos_filtrados = [item for item in documentos_eventos if item[1] == "PET"]

    if not documentos_eventos_filtrados:
        print("❌ Nenhuma petição encontrada.")
        return None, None

    # === 3. Pega a petição mais recente (maior evento_id) ===
    documentos_eventos_filtrados.sort(key=lambda x: int(x[2]), reverse=True)
    documento_requerido = documentos_eventos_filtrados[0]
    evento_id = documento_requerido[2]

    # Caminho onde o PDF será salvo
    caminho_pdf = os.path.join(pasta_downloads, f"{num_processo}.pdf")

    # Localiza o elemento do documento e o link
    elemento = navegador.find_element(By.ID, documento_requerido[0])
    link = elemento.find_element(By.XPATH, ".//*[contains(@class, 'infraLinkDoc')]")

    # Limpa histórico de requests para capturar apenas a próxima
    navegador.requests.clear()

    # Clica no link para gerar a requisição ao controlador.php
    link.click()
    
    # Aguarda o carregamento do preview/divBoxPreview para garantir que o clique gerou a requisição
    WebDriverWait(navegador, 10).until(
        EC.presence_of_element_located((By.ID, "divBoxPreview"))
    )
    time.sleep(2)  # Pequeno delay extra para garantir a requisição
    # Clica no botão "open-button" para abrir o documento em nova janela/aba
    # Vai para a aba que foi aberta
    if len(navegador.window_handles) > 1:
        navegador.switch_to.window(navegador.window_handles[-1])
    try:
        timestamp_download = datetime.now().isoformat()
        navegador.find_element(By.TAG_NAME, 'body').send_keys('\t\t\n')
        time.sleep(1)
    except Exception as e:
        print("Botão 'open-button' não encontrado ou erro ao clicar:", e)
    # Fecha a nova aba/janela aberta pelo botão "open-button"
    if len(navegador.window_handles) > 1:
        # Fecha todas as abas, menos a primeira
        while len(navegador.window_handles) > 1:
            navegador.switch_to.window(navegador.window_handles[-1])
            navegador.close()
        navegador.switch_to.window(navegador.window_handles[0])
    # === 4. Procura o PDF mais recente na pasta de downloads ===

    arquivos_pdf = glob.glob(os.path.join(pasta_downloads, "*.pdf"))
    if not arquivos_pdf:
        print("❌ Nenhum PDF encontrado na pasta de downloads.")
        return evento_id, None

    # Encontra o PDF mais recente
    pdf_mais_recente = max(arquivos_pdf, key=os.path.getmtime)
    tempo_modificacao = os.path.getmtime(pdf_mais_recente)
    tempo_modificacao_dt = datetime.fromtimestamp(tempo_modificacao)
    tempo_diff = abs((tempo_modificacao_dt - datetime.fromisoformat(timestamp_download)).total_seconds())

    if tempo_diff <= 10:
        # Move o arquivo para o caminho_pdf
        os.makedirs(os.path.dirname(caminho_pdf), exist_ok=True)
        os.replace(pdf_mais_recente, caminho_pdf)
        print(f"PDF renomeado e salvo em: {caminho_pdf}")

        return evento_id, caminho_pdf
    else:
        print(f"PDF mais recente tem diferença de {tempo_diff:.2f} segundos da timestamp. Não será renomeado.")
        return evento_id, None

def extrair_texto_pdf(caminho_pdf):
    with open(caminho_pdf, 'rb') as arquivo:
        leitor = PyPDF2.PdfReader(arquivo)
        texto = ""
        for pagina in leitor.pages:
            texto += pagina.extract_text()
    return texto

def pega_texto_documento(navegador, documento):
    WebDriverWait(navegador, 20).until(
        EC.presence_of_element_located((By.ID, documento))
    )
    # 1. Localizar o elemento pelo ID
    elemento = navegador.find_element(By.ID, documento)
    # 2. Criar ActionChains para executar o mouse over
    actions = ActionChains(navegador)
    # Rolar a página para o elemento antes de mover o mouse
    navegador.execute_script("arguments[0].scrollIntoView(true); window.scrollBy(0, -150);", elemento)
    # Faz o mouseover em cima do texto link infraLinkDocumento do elemento
    link_doc = elemento.find_element(By.XPATH, ".//*[contains(@class, 'infraLinkDoc')]")
    actions.move_to_element(link_doc).perform()
    # 3. Aguardar para o hover ter efeito
    time.sleep(5)
    
    # Verifica se há uma div com a classe 'divBoxPreview' visível na página
    overlays = navegador.find_elements(By.ID, "divBoxPreview")
    visiveis = [div for div in overlays if div.is_displayed()]

    conteudo = ""
    if visiveis:
        div = overlays[0]
        # Move o foco para a div
        ActionChains(navegador).move_to_element(div).click().perform()
        # Aguarda carregar o conteúdo (ajuste o tempo se necessário)
        time.sleep(1)
        # Seleciona o texto
        conteudo = navegador.find_element(By.ID, "divBoxPreview").text
        # Clica no botão de fechar o preview, se existir
        btn_close = navegador.find_element(By.ID, "divClosePreview")
        btn_close.click()    
        time.sleep(3)

    else:
        print("Erro ao recuperar o documento")
    return conteudo

def ollama_resumo(pedido):    
    pergunta_gemma = "Considere o seguinte pedido." \
    f"{pedido}" \
    "Resuma, da maneira mais objetiva possível, o pedido. Não mencione dados pessoais, como nomes, números de documento, números de processo, valores, etc. " \
    "O resumo deve ser genérico e breve (uma frase apenas, com o mínimo de palavras possível). " \
    "Se tiver mais de um pedido, retorne uma frase para cada um." \

    resumo = ollama.chat(
        model="cnmoro/gemma3-gaia-ptbr-4b:q4_k_m",
        messages=[{'role': 'user', 'content': f'{pergunta_gemma}'}],    
    )

    return(resumo['message']['content'])

def verifica_tipos_de_pedidos(pedido, lista_de_pedidos):

    print("========== Verificando se é um caso de uso conhecido... ==========")
    pergunta_gemma = "Considere a seguinte lista de pedidos:" \
    f"{lista_de_pedidos}" \
    f"É possível dizer que o pedido '{pedido}' pode ser adequadamente descrito por um item dessa lista?." \
    "Se sim, retorne APENAS o texto EXATO do resumo do pedido correspondente na lista. Se não, retorne APENAS o texto 'Não'."

    resumo = ollama.chat(
        model="cnmoro/gemma3-gaia-ptbr-4b:q8_0",
        messages=[{'role': 'user', 'content': f'{pergunta_gemma}'}],    
    )

    return(resumo['message']['content'])

def baixa_peticao_processo(navegador, processo):
    navegador.switch_to.default_content()
    eproc.entrar_no_processo(navegador, processo)

    with sqlite3.connect("urcaciv.db") as conn:
        evento_id, texto_peticao = pega_ultima_peticao(navegador, processo)

        print(f"========== TEXTO DA PETIÇÃO ==========")
        print(texto_peticao)


        with closing(conn.cursor()) as cursor:
            cursor.execute(
                f"UPDATE {perfil} SET pet = ? WHERE num_processo = ?",    
                (texto_peticao, processo)
            )
            conn.commit()
        






In [7]:
# Apaga os campos "resumo" onde "pet" é igual a "ERRO"

with sqlite3.connect("urcaciv.db") as conn:
    with closing(conn.cursor()) as cursor:
        cursor.execute(
            f"UPDATE {perfil} SET resumo = NULL WHERE pet = 'ERRO'"
        )
        conn.commit()
        print(f"Campos 'resumo' apagados onde 'pet' = 'ERRO' na tabela {perfil}")

Campos 'resumo' apagados onde 'pet' = 'ERRO' na tabela STM1CIV


In [6]:
#EXECUTA! resume as petições pendentes

perfil = "STM1CIV"

with sqlite3.connect("urcaciv.db") as conn:
    query = f"SELECT * FROM {perfil} WHERE pet IS NOT NULL AND (resumo IS NULL OR resumo = '')"
    pendentes = conn.execute(query).fetchall()

i=0
for processo in pendentes:
    i+=1
    texto_peticao = processo[4]
    num_processo = processo[1]

    print(f"Processo {i} de {len(pendentes)}: {num_processo}. Bytes: {len(texto_peticao.encode('utf-8'))}")
    #print("==== TEXTO DA PETIÇÃO ======")
    #print(texto_peticao)
    print("==== gerando resumo... ======")
    if len(texto_peticao.encode('utf-8')) > 10000:
        resumo_ollama = "Texto maior do que 10.000 bytes, provavelmente pedido complexo"
    else:
        resumo_ollama = ollama_resumo(texto_peticao)
    print("========== RESUMO ==========")
    print(resumo_ollama)
    print("============================")

    with closing(conn.cursor()) as cursor:
        cursor.execute(
            f"UPDATE {perfil} SET resumo = ? WHERE num_processo = ?",    
            (resumo_ollama, num_processo)
        )
        conn.commit()



Processo 1 de 32: 5044728-61.2024.8.21.0027. Bytes: 4
==== gerando resumo... ======
========== RESUMO ==========
Ok, estou pronto. Por favor, forneça o pedido.

Processo 2 de 32: 5001034-04.2008.8.21.0027 . Bytes: 4
==== gerando resumo... ======
========== RESUMO ==========
Ok, preciso resumir o pedido dado de forma objetiva e concisa, sem incluir quaisquer detalhes específicos que possam comprometer a privacidade ou o conteúdo original.
Para fornecer uma resposta precisa, preciso saber o que é o pedido. Aguarde, o prompt em si está pedindo-me para resumir um pedido. Isso é um pouco auto-referencial.

Assumindo que o pedido original é algo como "Resumir o seguinte texto/documento", aqui está o resumo:

Gere um resumo objetivo e conciso do conteúdo fornecido.

Processo 3 de 32: 5000983-07.2019.8.21.0027. Bytes: 4
==== gerando resumo... ======
========== RESUMO ==========
Por favor, forneça o pedido para que eu possa resumir de acordo.

Processo 4 de 32: 5045953-19.2024.8.21.0027. Bytes:

KeyboardInterrupt: 